# Loyiha 2: Gradient Descent noldan — batch, SGD, mini-batch — California Housing (1990)

**Talaba:** student3 · **Modul:** M2 (Calculus va GD) + M1 · **Kitob (ML_GUIDE):** 3-bob (GD, Batch/Stochastic/Mini-batch) · **Qiyinlik:** ⭐⭐⭐⭐ · **Taxminiy vaqt:** 8–10 soat · **Muddat:** ____________
> **Qoidalar.** Bu notebook — sizning ish daftaringiz. Har bir `# TODO` katakni to'ldiring, har bir **Savol** ga darhol pastida yangi *markdown* katak ochib 2–5 gap bilan javob bering ("ha/yo'q" baholanmaydi). Topshirishdan oldin `Kernel → Restart & Run All` — notebook yuqoridan pastga **xatosiz** ishlashi shart. Internetdan o'rganish — mumkin va kerak; tayyor kodni nusxalash — yo'q (himoyada har bir qatorni tushuntirib bera olishingiz kerak).

## Loyiha haqida

Normal Equation "bir zarbada" yechadi, lekin million qatorli yoki chuqur modellar uchun ishlamaydi — u yerda **faqat Gradient Descent**. Bu loyihada GD ni o'z sinfingiz (`class`) sifatida yozasiz va uning xulqini real ma'lumotda o'rganasiz: learning rate, scaling, batch o'lchami, vaqt, shovqin.

**Nimani o'rganasiz:** MSE gradienti `(2/m)·Xᵀ(Xθ − y)` · epoch vs iteratsiya · loss egri chizig'i · nega scaling GD uchun hayotiy · SGD shovqini va learning-rate schedule · sklearn `SGDRegressor` bilan solishtirish.

**Taqiqlar:** GD ni faqat o'zingiz yozasiz. `sklearn` — split, scaler va yakuniy solishtirish uchun.

## Dataset: California Housing (1990)

20 640 ta Kaliforniya tumani (1990 yil aholi ro'yxati): o'rtacha daromad, uy yoshi, xonalar, aholi, koordinatalar → **uyning median narxi** (100 000 $). Bu **katta** dataset — Normal Equation hali ishlaydi, lekin GD ning batch/mini-batch farqi endi haqiqiy vaqtda seziladi.

**Manba:** `sklearn.datasets.fetch_california_housing` (birinchi chaqiruvda ~1 MB yuklaydi). Asl: StatLib, Pace & Barry 1997

| Ustun | Ma'nosi |
|---|---|
| MedInc | tuman median daromadi (10 000 $) |
| HouseAge | uylar median yoshi |
| AveRooms, AveBedrms | uy boshiga o'rtacha xona / yotoqxona — **max 141 / 34 (!)** |
| Population, AveOccup | aholi, uy boshiga odam — **max 1243 (!)** |
| Latitude, Longitude | koordinatalar |
| **MedHouseVal** | median narx, 100 000 $ — target, **5.0 da kesilgan** (992 ta) |

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing
d = fetch_california_housing(as_frame=True)
df = d.frame                                   # 20640 tuman
FEATURES, TARGET = list(d.feature_names), 'MedHouseVal'
print(df.shape); df.head()

(20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 1-vazifa: Tayyorlash
Missing yo'q, hammasi raqamli. Lekin `describe()` ga qarang: `AveRooms` max 141, `AveOccup` max 1243 — bular tuman emas, mehmonxona/talabalar yotoqxonasi. Hozircha **qoldiring** (7-vazifada qaytamiz). `Population` 3 dan 35 000 gacha — scaling shart.

Keyin: `train_test_split(test_size=0.2, random_state=42)` → `StandardScaler` **faqat train ga fit** → `X_train_s, X_test_s`. Scale qilinmagan nusxani ham saqlab qo'ying (6-vazifa uchun).

**Savol 1.** Feature'larning min/max oralig'ini jadval qiling. Eng katta va eng kichik oraliq necha marta farq qiladi?

Eng katta range/ eng kichik range = population / Latitude = 35679/9.41= 3791.6

In [2]:
df.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


In [5]:
X = df[FEATURES].values; y = df[TARGET].values
# TODO: split, scaler (faqat train ga fit)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [21]:
min_max = {}
for feature in FEATURES:
    min_max[feature] = np.max(df[feature])-np.min(df[feature])
    print(f"Feature: {feature} | Minimum: {np.min(df[feature]):.2f} | Maximum: {np.max(df[feature]):.2f} | Range: {np.max(df[feature])-np.min(df[feature]):.2f}")

print(np.round(max(min_max.values())/min(min_max.values()), 2))

Feature: MedInc | Minimum: 0.50 | Maximum: 15.00 | Range: 14.50
Feature: HouseAge | Minimum: 1.00 | Maximum: 52.00 | Range: 51.00
Feature: AveRooms | Minimum: 0.85 | Maximum: 141.91 | Range: 141.06
Feature: AveBedrms | Minimum: 0.33 | Maximum: 34.07 | Range: 33.73
Feature: Population | Minimum: 3.00 | Maximum: 35682.00 | Range: 35679.00
Feature: AveOccup | Minimum: 0.69 | Maximum: 1243.33 | Range: 1242.64
Feature: Latitude | Minimum: 32.54 | Maximum: 41.95 | Range: 9.41
Feature: Longitude | Minimum: -124.35 | Maximum: -114.31 | Range: 10.04
3791.6


## 2-vazifa: `LinearRegressionGD` sinfi
Talablar: `fit(X, y)` — `batch_size=None` (batch), `1` (SGD), `k` (mini-batch); har epoch da o'rtacha loss `self.loss_history` ga yoziladi; `predict(X)`; bias alohida `self.b` yoki X ga 1-lar ustuni — o'zingiz tanlang, lekin izohlang. Har epoch boshida ma'lumotni `np.random.permutation` bilan aralashtiring.

**Savol 2.** Gradient formulasini docstring ga yozing va uni 1 gap bilan izohlang: nega `Xᵀ(Xθ − y)` — bu har bir feature bo'yicha xatoning "yig'indisi"?

In [ ]:
class LinearRegressionGD:
    def __init__(self, lr=0.01, n_epochs=100, batch_size=None, seed=42):
        self.lr, self.n_epochs, self.batch_size, self.seed = lr, n_epochs, batch_size, seed
        self.bias = None
        self.weights = None
        self.loss_history = []

    def fit(self, X, y):
        """∇MSE = (2/m) Xᵀ(Xθ − y). batch_size=None → batch GD, 1 → SGD, k → mini-batch"""
        # TODO
        n_samples, n_features = X.shape

        weights = np.zeros(n_features)
        bias = 0.0

        for epoch in range(self.n_epochs):
            y_pred = X @ weights + bias
            error = y-y_pred
            # computing gradients 
            dw = (-2/n_samples)*X.T@error
            db = (-2/n_samples)*np.sum(error)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            loss = error**2
            self.loss_history.append(loss)
        
        return self

    def predict(self, X):
        return X*self.weights + self.bias

## 3-vazifa: Learning rate tajribasi (batch GD)
`lr ∈ {0.001, 0.01, 0.1, 0.5, 1.0}`, `n_epochs=200`. Barcha loss egri chiziqlarini **bitta** grafikda (`plt.yscale('log')`) chizing. Qaysi lr da divergensiya (loss o'sadi / `inf` / `nan`)?

**Savol 3.** Divergensiya nega aynan katta lr da bo'ladi? Kosa (bowl) metaforasi bilan tushuntiring (M2, 4-qism). Eng yaxshi lr ni qanday tanlaysiz — faqat "eng tez" mi?

In [ ]:
# TODO: 5 xil lr, loss_history grafigi, log-scale

## 4-vazifa: Batch vs SGD vs Mini-batch
Bir xil `lr` (3-vazifadagi eng yaxshisi) va `n_epochs=50`: `batch_size=None`, `1`, `32`, `256`. Har biri uchun: vaqt (`time.perf_counter`), oxirgi loss, test R². Loss egri chiziqlarini bitta grafikda chizing.

**Savol 4.** SGD egri chizig'i nega "titraydi"? Mini-batch nega amalda standart (kitob 3-bob jadvali)? Vaqt bo'yicha kutilmagan natija bo'ldimi — nega (maslahat: NumPy vektorlash vs Python sikli)?

In [ ]:
# TODO: 4 xil batch_size: vaqt, loss, test R2, grafik

## 5-vazifa: Normal Equation va sklearn bilan solishtirish
Uch ustunli jadval: sizning GD (eng yaxshi sozlama) · Normal Equation (P1 dagi funksiyangiz) · `SGDRegressor(max_iter=1000, tol=None, random_state=42)`. θ lar va test RMSE/R².

**Savol 5.** GD θ si Normal Equation dan qanchalik farq qiladi (3 kasrgacha)? Epoch sonini 5× oshirsangiz farq kamayadimi? Bu "yaqinlashish" (convergence) haqida nima deydi?

In [ ]:
# TODO: GD vs NE vs SGDRegressor jadvali

## 6-vazifa: Scaling bo'lmasa nima bo'ladi
1-vazifadagi **scale qilinmagan** X bilan 3-vazifadagi eng yaxshi lr da GD ni ishga tushiring. Nima bo'ldi? Endi `lr` ni kamaytirib (masalan 1e-5, 1e-6, 1e-7) ishlaydigan qiymat toping va necha epoch kerak bo'lganini ayting.

**Savol 6.** Scale qilinmagan ma'lumotda cost "kosa" si qanday shaklda (cho'zilgan ellips)? Nega bu GD ni sekinlashtiradi, Normal Equation ga esa ta'sir qilmaydi? (kitob 3-bob; M2 6-qism)

In [ ]:
# TODO: scale qilinmagan GD: divergensiya, kichik lr, epoch soni

## 7-vazifa: Learning-rate schedule
`lr_t = lr0 / (1 + k·t)` (t — epoch) ni sinfga qo'shing (`decay=k` parametri). SGD (`batch_size=1`) uchun schedule bilan va usiz loss egri chiziqlarini solishtiring.

**Outlier va GD.** `AveRooms`, `AveBedrms`, `AveOccup`, `Population` ga `np.log1p` qo'llang (split dan oldin, qator-ichi amal) va GD ni qayta o'rgating: R² 0.58 dan qanchaga o'sdi? Loss egri chizig'i tezroq tekislandimi? Nega bir nechta ekstremal qator scaled ma'lumotda ham gradientni "tortadi" (MSE kvadrat — katta xato kvadratda yanada katta)?

**Savol 7.** Schedule SGD ning oxirgi loss ini yaxshiladimi? Nega SGD ga schedule batch GD dan ko'ra ko'proq kerak?

In [ ]:
# TODO: decay parametri, SGD schedule bilan/usiz

## 🔍 Tadqiqot (majburiy — kamida 2 tasi)
1. **Momentum**: `v = β·v − lr·∇; θ = θ + v` (β=0.9) ni sinfga qo'shing. Loss egri chiziqlarini oddiy GD bilan solishtiring. Momentum nima uchun "tezlashtiradi" — 3 gap.
2. **Early stopping**: train ning 20% ini validation qilib ajrating; validation loss 10 epoch ketma-ket yaxshilanmasa to'xtang va eng yaxshi θ ni qaytaring. Qaysi epoch da to'xtadi?
3. `SGDRegressor` hujjatidan `eta0`, `learning_rate='invscaling'`, `power_t`, `tol`, `n_iter_no_change` parametrlarini o'rganing va 2 xil sozlama bilan solishtiring — sizning sinfingizdagi qaysi g'oyalarga mos keladi?
4. **Vaqt va hajm.** Train ning 2 000 / 8 000 / 16 500 qatori bilan (`np.random.choice`) batch GD va mini-batch (32) ni `n_epochs=20` da o'rgating — har biri necha soniya, yakuniy loss qancha? Bitta epoch da nechta θ yangilanishi bo'ladi? Grafik: x = m, y = vaqt. Qaysi rejim m ga proporsional o'sadi? Bonus: `Latitude` × `Longitude` corr ≈ −0.93 — nega (xaritaga qarang) va bu GD ga xalaqit beradimi?

In [ ]:
# TODO: tadqiqot

## ⭐ Bonus (+10)
Faqat 2 ta feature tanlab (bias yo'q, scale qilingan), cost funksiyasining **contour** grafigini chizing (M3, 1-qism) va batch GD hamda SGD ning θ yo'lini (har epoch dagi θ) ustiga chizing. Ikkala yo'l qanday farq qiladi?

## Topshirish

- Fayl nomi: `student3_P2.ipynb` (shu fayl, to'ldirilgan). `Restart & Run All` qilingan, barcha grafiklar ko'rinib turibdi.
- Oxirgi katak — **Xulosa** (markdown, 6–10 gap): nimani o'rgandingiz, qaysi natija kutilmagan bo'ldi, nimani hali tushunmadingiz.
- Himoya: 5 daqiqa — istalgan katakdagi kodni tushuntirib berasiz.

## Baholash (100 ball + bonus)

| Mezon | Ball |
|---|---|
| `LinearRegressionGD` sinfi to'g'ri (3 rejim, loss_history, aralashtirish) | 25 |
| 3–7 vazifalar tajribalari va jadvallar | 25 |
| Savollarga tahliliy javoblar (7 ta) | 20 |
| Tadqiqot (2 ta) | 15 |
| Grafiklar + kod tozaligi | 15 |
| ⭐ Bonus | +10 |